In [1]:
# imports
import pandas as pd
import matplotlib as plt
import numpy as np
import math

# display options
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
def fmtCode(code, length, withInt=False):
    if isinstance(code, float) and math.isnan(code):
        return "NA"
    if withInt:
        if isinstance(code, str) and not code.isdigit():
            return code
        return "0" * (length - len(str(int(code)))) + str(int(code))
    return "0" * (length - len(str(code))) + str(code)

def getBackupValues(row, names):
    for name in names[:-1]:
        if not pd.isna(row[name]):
            return row[name]
    return row[names[-1]]

def getDeptNumber(row):
    return getBackupValues(row, ["department_code", "department_number"])

def getAppropriationAuthority(row):
    return getBackupValues(row, ["org", "appropriation_authority"])

def getAppropriationAmount(row):
    return getBackupValues(row, ["appropration", "_appropriation", "appropriation"])

def getRevisedAppropriationAmount(row):
    return getBackupValues(row, ["_revised_appropriation", "revised_appropriation"])

def getRecommendationAmount(row):
    return getBackupValues(row, ["_recommendation", "recommendation"])

In [3]:
years = list(range(2012, 2026))
df = pd.DataFrame()

for year in years:
    dfYear = pd.read_csv(f"../data/{year}-recommendations.csv")
    dfYear["year"] = year
    df = pd.concat([df, dfYear], ignore_index=True)

# clean rows and columns
df["fund_code"] = df.apply(lambda row: fmtCode(row["fund_code"], 4), axis=1)
df["department_number"] = df.apply(
    lambda row: fmtCode(getDeptNumber(row), 2, True), axis=1
)
df["appropriation_account"] = df["appropriation_account"].apply(
    lambda x: fmtCode(x, 4, True)
)
df["appropriation_authority"] = df.apply(
    lambda row: fmtCode(getAppropriationAuthority(row), 4, True), axis=1
)
df["appropriation"] = df.apply(getAppropriationAmount, axis=1)
df["revised_appropriation"] = df.apply(getRevisedAppropriationAmount, axis=1)
df["recommendation"] = df.apply(getRecommendationAmount, axis=1)
df = df[
    [
        "year",
        "department_number",
        "fund_code",
        "appropriation_account",
        "appropriation_authority",
        "appropriation",
        "revised_appropriation",
        "recommendation",
    ]
]
df.head()

,year,department_number,fund_code,appropriation_account,appropriation_authority,appropriation,revised_appropriation,recommendation
0,2012,01,0100,0229,2005,680.00,680.00,680.00
1,2012,01,0100,0270,2005,872.00,872.00,872.00
2,2012,01,0100,0150,2005,1242.00,1242.00,1000.00
3,2012,01,0100,0126,2005,1310.00,1310.00,1200.00
4,2012,01,0100,0196,2005,0.00,0.00,3800.00


In [4]:
df[
    ~pd.isna(df["appropriation"])
    & ~pd.isna(df["revised_appropriation"])
    & (df["appropriation"] != df["revised_appropriation"])
]

,year,department_number,fund_code,appropriation_account,appropriation_authority,appropriation,revised_appropriation,recommendation
85,2012,15,0100,0700,2005,48000.00,43000.00,43000.00
87,2012,15,0100,9010,2005,107072.00,92072.00,92072.00
94,2012,15,0100,9071,2005,114232.00,0.00,0.00
97,2012,15,0100,0169,2010,5000.00,2500.00,2500.00
101,2012,15,0100,0229,2010,10000.00,8000.00,8000.00
102,2012,15,0100,0140,2010,20000.00,12500.00,12500.00
103,2012,15,0100,0130,2010,20000.00,16000.00,16000.00
108,2012,15,0100,9005,2010,100000.00,50000.00,50000.00
110,2012,15,0100,9010,2010,100000.00,92500.00,92500.00
111,2012,15,0100,0005,2010,1672704.00,1637704.00,1698740.00
